In [1]:
# Verify PostgreSQL database setup
This notebook validates the database connection module and schema setup script by creating the required files and running `setup_db.py`.
</VSCode.Cell>
<VSCode.Cell language="python">
# Section 1: Import Required Libraries and Environment Variables
import os
import logging
from contextlib import contextmanager

from dotenv import load_dotenv

# Load environment variables from .env at the project root
load_dotenv(dotenv_path="/workspaces/Saudi-stocks/.env")
DATABASE_URL = os.getenv("DATABASE_URL")
print(f"Loaded DATABASE_URL: {DATABASE_URL}")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
</VSCode.Cell>
<VSCode.Cell language="python">
# Section 2: Implement DatabaseConnection
import psycopg2

class DatabaseConnection:
    def __init__(self):
        self.database_url = os.getenv("DATABASE_URL")
        if not self.database_url:
            raise RuntimeError("DATABASE_URL is not set in the environment.")

    def get_connection(self):
        try:
            conn = psycopg2.connect(self.database_url)
            conn.autocommit = False
            return conn
        except Exception:
            logging.exception("Failed to connect to the database.")
            raise

    @contextmanager
    def connection(self):
        conn = None
        try:
            conn = self.get_connection()
            yield conn
            conn.commit()
        except Exception:
            if conn is not None:
                conn.rollback()
            raise
        finally:
            if conn is not None:
                conn.close()

    def execute_query(self, query, params=None):
        with self.connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute(query, params)

    def fetch_all(self, query, params=None):
        with self.connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute(query, params)
                return cursor.fetchall()

print("DatabaseConnection class defined.")
</VSCode.Cell>
<VSCode.Cell language="python">
# Section 3: Implement SchemaManager and Table Definitions

class SchemaManager:
    def __init__(self, db_connection: DatabaseConnection):
        self.db = db_connection

    def create_all_tables(self):
        statements = [
            self._create_companies_table(),
            self._create_daily_prices_table(),
            self._create_financials_table(),
            self._create_dividends_table(),
            self._create_announcements_table(),
            self._create_news_table(),
        ]

        for statement in statements:
            self.db.execute_query(statement)

    def drop_all_tables(self):
        statements = [
            "DROP TABLE IF EXISTS news CASCADE;",
            "DROP TABLE IF EXISTS announcements CASCADE;",
            "DROP TABLE IF EXISTS dividends CASCADE;",
            "DROP TABLE IF EXISTS financials CASCADE;",
            "DROP TABLE IF EXISTS daily_prices CASCADE;",
            "DROP TABLE IF EXISTS companies CASCADE;",
        ]
        for statement in statements:
            self.db.execute_query(statement)

    def verify_tables(self):
        tables = [
            "companies",
            "daily_prices",
            "financials",
            "dividends",
            "announcements",
            "news",
        ]
        for table in tables:
            row_count = self._get_row_count(table)
            print(f"{table}: {row_count} rows")

    def _get_row_count(self, table_name: str) -> int:
        exists_result = self.db.fetch_all(
            "SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'public' AND table_name = %s;",
            (table_name,),
        )
        if exists_result and exists_result[0][0] == 1:
            count_result = self.db.fetch_all(f"SELECT COUNT(*) FROM {table_name};")
            return count_result[0][0] if count_result else 0
        return 0

    def _create_companies_table(self) -> str:
        return (
            "CREATE TABLE IF NOT EXISTS companies ("
            "symbol VARCHAR(10) PRIMARY KEY, "
            "name_ar VARCHAR(255), "
            "name_en VARCHAR(255), "
            "sector_ar VARCHAR(100), "
            "sector_en VARCHAR(100), "
            "share_type VARCHAR(50), "
            "total_shares BIGINT, "
            "paid_capital BIGINT, "
            "isin VARCHAR(20), "
            "founded_date DATE, "
            "fiscal_year_end VARCHAR(5), "
            "auditor VARCHAR(100), "
            "market_cap DECIMAL(15,2), "
            "company_url VARCHAR(500), "
            "last_updated TIMESTAMP DEFAULT NOW()"
            ");"
        )

    def _create_daily_prices_table(self) -> str:
        return (
            "CREATE TABLE IF NOT EXISTS daily_prices ("
            "symbol VARCHAR(10), "
            "trade_date DATE, "
            "open_price DECIMAL(10,2), "
            "high_price DECIMAL(10,2), "
            "low_price DECIMAL(10,2), "
            "close_price DECIMAL(10,2), "
            "prev_close DECIMAL(10,2), "
            "change_value DECIMAL(10,2), "
            "change_pct DECIMAL(6,2), "
            "volume BIGINT, "
            "value_traded DECIMAL(15,2), "
            "num_trades INT, "
            "best_bid_price DECIMAL(10,2), "
            "best_bid_qty INT, "
            "best_offer_price DECIMAL(10,2), "
            "best_offer_qty INT, "
            "PRIMARY KEY (symbol, trade_date)"
            ");"
        )

    def _create_financials_table(self) -> str:
        return (
            "CREATE TABLE IF NOT EXISTS financials ("
            "symbol VARCHAR(10), "
            "period_end DATE, "
            "period_type VARCHAR(10), "
            "total_revenue DECIMAL(15,2), "
            "net_profit_before_zakat DECIMAL(15,2), "
            "zakat_tax DECIMAL(15,2), "
            "net_profit DECIMAL(15,2), "
            "eps DECIMAL(10,2), "
            "total_assets DECIMAL(15,2), "
            "total_liabilities DECIMAL(15,2), "
            "total_equity DECIMAL(15,2), "
            "operating_cash_flow DECIMAL(15,2), "
            "investing_cash_flow DECIMAL(15,2), "
            "financing_cash_flow DECIMAL(15,2), "
            "cash_end DECIMAL(15,2), "
            "PRIMARY KEY (symbol, period_end, period_type)"
            ");"
        )

    def _create_dividends_table(self) -> str:
        return (
            "CREATE TABLE IF NOT EXISTS dividends ("
            "symbol VARCHAR(10), "
            "announcement_date DATE, "
            "eligibility_date DATE, "
            "distribution_date DATE, "
            "amount_per_share DECIMAL(10,2), "
            "PRIMARY KEY (symbol, eligibility_date)"
            ");"
        )

    def _create_announcements_table(self) -> str:
        return (
            "CREATE TABLE IF NOT EXISTS announcements ("
            "id SERIAL PRIMARY KEY, "
            "symbol VARCHAR(10), "
            "announcement_date TIMESTAMP, "
            "title_ar TEXT, "
            "price_on_date DECIMAL(10,2), "
            "change_pct_on_date DECIMAL(6,2)"
            ");"
        )

    def _create_news_table(self) -> str:
        return (
            "CREATE TABLE IF NOT EXISTS news ("
            "id SERIAL PRIMARY KEY, "
            "source VARCHAR(100), "
            "published_date TIMESTAMP, "
            "title TEXT, "
            "summary TEXT, "
            "url VARCHAR(500) UNIQUE, "
            "sentiment_score DECIMAL(4,2), "
            "related_symbols TEXT, "
            "fetched_at TIMESTAMP DEFAULT NOW()"
            ");"
        )

print("SchemaManager class defined.")
</VSCode.Cell>
<VSCode.Cell language="python">
# Section 4: Create db Package Exports
import pathlib

package_dir = pathlib.Path("/workspaces/Saudi-stocks/db")
package_dir.mkdir(parents=True, exist_ok=True)

init_file = package_dir / "__init__.py"
init_file.write_text(
"from .connection import DatabaseConnection\nfrom .schema import SchemaManager\n\n__all__ = ['DatabaseConnection', 'SchemaManager']\n"
)
print(f"Created {init_file}")
</VSCode.Cell>
<VSCode.Cell language="python">
# Section 5: Write setup_db.py Script
setup_path = pathlib.Path("/workspaces/Saudi-stocks/setup_db.py")
setup_path.write_text(
"from db.connection import DatabaseConnection\n"
"from db.schema import SchemaManager\n\n"
"def main():\n"
"    db = DatabaseConnection()\n"
"    schema = SchemaManager(db)\n\n"
"    print('Creating database tables...')\n"
"    schema.create_all_tables()\n\n"
"    print('Verifying created tables...')\n"
"    schema.verify_tables()\n\n"
"    print('Database setup complete. All tables are created and verified.')\n\n"
"if __name__ == '__main__':\n"
"    main()\n"
)
print(f"Created {setup_path}")
</VSCode.Cell>
<VSCode.Cell language="python">
# Section 6: Run setup_db.py to Verify Tables
import subprocess

print('Running setup_db.py...')
result = subprocess.run(
    ["python3", "/workspaces/Saudi-stocks/setup_db.py"],
    capture_output=True,
    text=True,
)
print('STDOUT:')
print(result.stdout)
print('STDERR:')
print(result.stderr)
print('Return code:', result.returncode)


SyntaxError: invalid syntax (1677778813.py, line 2)